In [0]:
%%writefile doctors.csv
doctor_id,doctor_name,specialization,city,experience_years,consultation_fee
D101,Dr. Ramesh,Cardiology,Hyderabad,12,1500
D102,Dr. Priya,Neurology,Bangalore,10,2000
D103,Dr. Anita,Dermatology,Chennai,8,1000
D104,Dr. Suresh,Orthopedics,Mumbai,15,2500
D105,Dr. Meera,Pediatrics,Delhi,6,1200
D106,Dr. Kiran,Cardiology,Hyderabad,20,3000
D107,Dr. Farhan,Neurology,Pune,5,1800
D108,Dr. Nisha,Dermatology,Kochi,9,1100

Overwriting doctors.csv


In [0]:
%%writefile visits.csv
visit_id,patient_name,doctor_id,visit_date,diagnosis,bill_amount,payment_status
V1001,Rahul Sharma,D101,2026-01-10,Heart Checkup,5000,Paid
V1002,Priya Reddy,D102,2026-01-10,Migraine,3500,Paid
V1003,Amit Kumar,D103,2026-01-11,Skin Allergy,2000,Pending
V1004,Sneha Patel,D104,2026-01-11,Fracture,12000,Paid
V1005,Farhan Ali,D105,2026-01-12,Fever,1500,Paid
V1006,Neha Singh,D106,2026-01-12,Heart Checkup,7000,Paid
V1007,Arjun Verma,D120,2026-01-13,Migraine,4000,Paid
V1008,Meera Nair,D103,2026-01-13,Skin Allergy,,Pending
V1009,Kiran Rao,D104,2026-01-14,Back Pain,6000,Paid
V1010,Nisha Reddy,D101,2026-01-14,Heart Checkup,5500,Paid

Overwriting visits.csv


In [0]:
%%writefile hospital_config.json
[
{
"hospital_id":"H101",
"hospital_name":"Apollo Hospital",
"city":"Hyderabad",
"contact":{"phone":"9876500011","email":"apollo@mail.com"},
"services":["Cardiology","Neurology","Dermatology"]
},
{
"hospital_id":"H102",
"hospital_name":"Yashoda Hospital",
"city":"Hyderabad",
"contact":{"phone":null,"email":"yashoda@mail.com"},
"services":["Cardiology","Orthopedics"]
},
{
"hospital_id":"H103",
"hospital_name":"Care Hospital",
"city":"Bangalore",
"contact":{"phone":"9876500013","email":null},
"services":["Neurology","Pediatrics"]
}
]

Overwriting hospital_config.json


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("HospitalAnalytics").getOrCreate()



In [0]:
doctors_df = spark.read.csv(
    "file:/Workspace/Users/atsthasneem@gmail.com/doctors.csv",
    header=True,
    inferSchema=True
)

doctors_df.show()

+---------+-----------+--------------+---------+----------------+----------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|
+---------+-----------+--------------+---------+----------------+----------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|            2000|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|            1000|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|            2500|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|               6|            1200|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|            3000|
|     D107| Dr. Farhan|     Neurology|     Pune|               5|            1800|
|     D108|  Dr. Nisha|   Dermatology|    Kochi|               9|            1100|
+---------+-----------+--------------+---------+----------------+----------------+



In [0]:
doctors_df.printSchema()


root
 |-- doctor_id: string (nullable = true)
 |-- doctor_name: string (nullable = true)
 |-- specialization: string (nullable = true)
 |-- city: string (nullable = true)
 |-- experience_years: integer (nullable = true)
 |-- consultation_fee: integer (nullable = true)



In [0]:
visits_df = spark.read.csv(
    "file:/Workspace/Users/atsthasneem@gmail.com/visits.csv",
    header=True,
    inferSchema=True
)

hospital_df = spark.read.json(
    "file:/Workspace/Users/atsthasneem@gmail.com/hospital_config.json"
)

In [0]:

visits_df.printSchema()
hospital_df.printSchema()

root
 |-- visit_id: string (nullable = true)
 |-- patient_name: string (nullable = true)
 |-- doctor_id: string (nullable = true)
 |-- visit_date: date (nullable = true)
 |-- diagnosis: string (nullable = true)
 |-- bill_amount: integer (nullable = true)
 |-- payment_status: string (nullable = true)

root
 |-- _corrupt_record: string (nullable = true)



In [0]:
hospital_df = spark.read \
    .option("multiline", "true") \
    .json("file:/Workspace/Users/atsthasneem@gmail.com/hospital_config.json")

hospital_df.printSchema()
hospital_df.show(truncate=False)

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- hospital_id: string (nullable = true)
 |-- hospital_name: string (nullable = true)
 |-- services: array (nullable = true)
 |    |-- element: string (containsNull = true)

+---------+-----------------------------+-----------+----------------+------------------------------------+
|city     |contact                      |hospital_id|hospital_name   |services                            |
+---------+-----------------------------+-----------+----------------+------------------------------------+
|Hyderabad|{apollo@mail.com, 9876500011}|H101       |Apollo Hospital |[Cardiology, Neurology, Dermatology]|
|Hyderabad|{yashoda@mail.com, NULL}     |H102       |Yashoda Hospital|[Cardiology, Orthopedics]           |
|Bangalore|{NULL, 9876500013}           |H103       |Care Hospital   |[Neurology, Pediatrics]             |
+---------+-

In [0]:
doctors_df.show()
visits_df.show()
hospital_df.show()

+---------+-----------+--------------+---------+----------------+----------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|
+---------+-----------+--------------+---------+----------------+----------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|            2000|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|            1000|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|            2500|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|               6|            1200|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|            3000|
|     D107| Dr. Farhan|     Neurology|     Pune|               5|            1800|
|     D108|  Dr. Nisha|   Dermatology|    Kochi|               9|            1100|
+---------+-----------+--------------+---------+----------------+----------------+

+--

In [0]:
doctors_df.count()

8

In [0]:
visits_df.count()

10

In [0]:
doctors_df.filter(col("city")=="Hyderabad").show()

+---------+-----------+--------------+---------+----------------+----------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|
+---------+-----------+--------------+---------+----------------+----------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|            3000|
+---------+-----------+--------------+---------+----------------+----------------+



In [0]:
doctors_df.filter(col("specialization")=="Cardiology").show()

+---------+-----------+--------------+---------+----------------+----------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|
+---------+-----------+--------------+---------+----------------+----------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|            3000|
+---------+-----------+--------------+---------+----------------+----------------+



In [0]:
doctors_df.filter(col("experience_years")>10).show()

+---------+-----------+--------------+---------+----------------+----------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|
+---------+-----------+--------------+---------+----------------+----------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|            2500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|            3000|
+---------+-----------+--------------+---------+----------------+----------------+



In [0]:
visits_df.filter(col("bill_amount")>5000).show()

+--------+------------+---------+----------+-------------+-----------+--------------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_status|
+--------+------------+---------+----------+-------------+-----------+--------------+
|   V1004| Sneha Patel|     D104|2026-01-11|     Fracture|      12000|          Paid|
|   V1006|  Neha Singh|     D106|2026-01-12|Heart Checkup|       7000|          Paid|
|   V1009|   Kiran Rao|     D104|2026-01-14|    Back Pain|       6000|          Paid|
|   V1010| Nisha Reddy|     D101|2026-01-14|Heart Checkup|       5500|          Paid|
+--------+------------+---------+----------+-------------+-----------+--------------+



In [0]:
visits_df.filter(col("payment_status")=="Pending").show()

+--------+------------+---------+----------+------------+-----------+--------------+
|visit_id|patient_name|doctor_id|visit_date|   diagnosis|bill_amount|payment_status|
+--------+------------+---------+----------+------------+-----------+--------------+
|   V1003|  Amit Kumar|     D103|2026-01-11|Skin Allergy|       2000|       Pending|
|   V1008|  Meera Nair|     D103|2026-01-13|Skin Allergy|       NULL|       Pending|
+--------+------------+---------+----------+------------+-----------+--------------+



In [0]:
visits_df.filter(col("payment_status")=="Paid").show()

+--------+------------+---------+----------+-------------+-----------+--------------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_status|
+--------+------------+---------+----------+-------------+-----------+--------------+
|   V1001|Rahul Sharma|     D101|2026-01-10|Heart Checkup|       5000|          Paid|
|   V1002| Priya Reddy|     D102|2026-01-10|     Migraine|       3500|          Paid|
|   V1004| Sneha Patel|     D104|2026-01-11|     Fracture|      12000|          Paid|
|   V1005|  Farhan Ali|     D105|2026-01-12|        Fever|       1500|          Paid|
|   V1006|  Neha Singh|     D106|2026-01-12|Heart Checkup|       7000|          Paid|
|   V1007| Arjun Verma|     D120|2026-01-13|     Migraine|       4000|          Paid|
|   V1009|   Kiran Rao|     D104|2026-01-14|    Back Pain|       6000|          Paid|
|   V1010| Nisha Reddy|     D101|2026-01-14|Heart Checkup|       5500|          Paid|
+--------+------------+---------+----------+----------

In [0]:
doctors_df.groupBy("specialization")\
.agg(avg("consultation_fee").alias("avg_fee")).show()


+--------------+-------+
|specialization|avg_fee|
+--------------+-------+
|   Orthopedics| 2500.0|
|    Cardiology| 2250.0|
|    Pediatrics| 1200.0|
|   Dermatology| 1050.0|
|     Neurology| 1900.0|
+--------------+-------+



In [0]:
doctors_df.groupBy("specialization").agg(max("consultation_fee").alias("max_fee")).show()

+--------------+-------+
|specialization|max_fee|
+--------------+-------+
|   Orthopedics|   2500|
|    Cardiology|   3000|
|    Pediatrics|   1200|
|   Dermatology|   1100|
|     Neurology|   2000|
+--------------+-------+



In [0]:
doctors_df.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|    Delhi|    1|
|  Chennai|    1|
|    Kochi|    1|
|Hyderabad|    2|
|Bangalore|    1|
|     Pune|    1|
|   Mumbai|    1|
+---------+-----+



In [0]:
doctors_df.groupBy("specialization").count().show()

+--------------+-----+
|specialization|count|
+--------------+-----+
|   Orthopedics|    1|
|    Cardiology|    2|
|    Pediatrics|    1|
|   Dermatology|    2|
|     Neurology|    2|
+--------------+-----+



In [0]:
from pyspark.sql.functions import sum as _sum; visits_df.agg(_sum("bill_amount")).show()

+----------------+
|sum(bill_amount)|
+----------------+
|           46500|
+----------------+



In [0]:
visits_df.agg(avg("bill_amount")).show()

+-----------------+
| avg(bill_amount)|
+-----------------+
|5166.666666666667|
+-----------------+



In [0]:
visits_df.agg(max("bill_amount")).show()

+----------------+
|max(bill_amount)|
+----------------+
|           12000|
+----------------+



In [0]:
visits_df.agg(min("bill_amount")).show()

+----------------+
|min(bill_amount)|
+----------------+
|            1500|
+----------------+



In [0]:
doctors_df.orderBy(col("consultation_fee").desc()).show()

+---------+-----------+--------------+---------+----------------+----------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|
+---------+-----------+--------------+---------+----------------+----------------+
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|            3000|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|            2500|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|            2000|
|     D107| Dr. Farhan|     Neurology|     Pune|               5|            1800|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|               6|            1200|
|     D108|  Dr. Nisha|   Dermatology|    Kochi|               9|            1100|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|            1000|
+---------+-----------+--------------+---------+----------------+----------------+



In [0]:
visits_df.orderBy(col("bill_amount").desc()).show()

+--------+------------+---------+----------+-------------+-----------+--------------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_status|
+--------+------------+---------+----------+-------------+-----------+--------------+
|   V1004| Sneha Patel|     D104|2026-01-11|     Fracture|      12000|          Paid|
|   V1006|  Neha Singh|     D106|2026-01-12|Heart Checkup|       7000|          Paid|
|   V1009|   Kiran Rao|     D104|2026-01-14|    Back Pain|       6000|          Paid|
|   V1010| Nisha Reddy|     D101|2026-01-14|Heart Checkup|       5500|          Paid|
|   V1001|Rahul Sharma|     D101|2026-01-10|Heart Checkup|       5000|          Paid|
|   V1007| Arjun Verma|     D120|2026-01-13|     Migraine|       4000|          Paid|
|   V1002| Priya Reddy|     D102|2026-01-10|     Migraine|       3500|          Paid|
|   V1003|  Amit Kumar|     D103|2026-01-11| Skin Allergy|       2000|       Pending|
|   V1005|  Farhan Ali|     D105|2026-01-12|        Fe

In [0]:
visits_df.filter(col("bill_amount").isNull()).show()

+--------+------------+---------+----------+------------+-----------+--------------+
|visit_id|patient_name|doctor_id|visit_date|   diagnosis|bill_amount|payment_status|
+--------+------------+---------+----------+------------+-----------+--------------+
|   V1008|  Meera Nair|     D103|2026-01-13|Skin Allergy|       NULL|       Pending|
+--------+------------+---------+----------+------------+-----------+--------------+



In [0]:
visits_df = visits_df.fillna({"bill_amount":0})

In [0]:
visits_df = visits_df.withColumn(
    "tax",
    col("bill_amount")*0.05
)

In [0]:
visits_df = visits_df.withColumn(
    "final_bill",
    col("bill_amount")+col("tax")
)

Join Exercises

In [0]:
inner_df = doctors_df.join(visits_df,"doctor_id","inner")

In [0]:
left_df = doctors_df.join(visits_df,"doctor_id","left")

In [0]:
right_df = doctors_df.join(visits_df,"doctor_id","right")

In [0]:
full_df = doctors_df.join(visits_df,"doctor_id","full")

In [0]:
visits_df.join(
    doctors_df,
    "doctor_id",
    "left_anti"
).show()

+---------+--------+------------+----------+---------+-----------+--------------+-----+----------+
|doctor_id|visit_id|patient_name|visit_date|diagnosis|bill_amount|payment_status|  tax|final_bill|
+---------+--------+------------+----------+---------+-----------+--------------+-----+----------+
|     D120|   V1007| Arjun Verma|2026-01-13| Migraine|       4000|          Paid|200.0|    4200.0|
+---------+--------+------------+----------+---------+-----------+--------------+-----+----------+



In [0]:
doctors_df.join(
    visits_df,
    "doctor_id",
    "left_anti"
).show()

+---------+-----------+--------------+-----+----------------+----------------+
|doctor_id|doctor_name|specialization| city|experience_years|consultation_fee|
+---------+-----------+--------------+-----+----------------+----------------+
|     D107| Dr. Farhan|     Neurology| Pune|               5|            1800|
|     D108|  Dr. Nisha|   Dermatology|Kochi|               9|            1100|
+---------+-----------+--------------+-----+----------------+----------------+



In [0]:
inner_df.groupBy(
    "doctor_id",
    "doctor_name"
).count().show()

+---------+-----------+-----+
|doctor_id|doctor_name|count|
+---------+-----------+-----+
|     D102|  Dr. Priya|    1|
|     D105|  Dr. Meera|    1|
|     D104| Dr. Suresh|    2|
|     D103|  Dr. Anita|    2|
|     D101| Dr. Ramesh|    2|
|     D106|  Dr. Kiran|    1|
+---------+-----------+-----+



In [0]:
joined_df = doctors_df.join(
    visits_df,
    "doctor_id",
    "inner"
)

joined_df.show()

+---------+-----------+--------------+---------+----------------+----------------+--------+------------+----------+-------------+-----------+--------------+-----+----------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|visit_id|patient_name|visit_date|    diagnosis|bill_amount|payment_status|  tax|final_bill|
+---------+-----------+--------------+---------+----------------+----------------+--------+------------+----------+-------------+-----------+--------------+-----+----------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|   V1001|Rahul Sharma|2026-01-10|Heart Checkup|       5000|          Paid|250.0|    5250.0|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|            2000|   V1002| Priya Reddy|2026-01-10|     Migraine|       3500|          Paid|175.0|    3675.0|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|            1000|   V1003|  Amit Kumar|2026-01-11| Skin Allergy| 

In [0]:
from pyspark.sql.functions import *

revenue_df = joined_df.groupBy(
    "doctor_id",
    "doctor_name",
    "specialization",
    "city"
).agg(
    sum("bill_amount").alias("revenue")
)

revenue_df.show()

+---------+-----------+--------------+---------+-------+
|doctor_id|doctor_name|specialization|     city|revenue|
+---------+-----------+--------------+---------+-------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|
+---------+-----------+--------------+---------+-------+



In [0]:
revenue_df.orderBy(
    col("revenue").desc()
).show(1)

+---------+-----------+--------------+------+-------+
|doctor_id|doctor_name|specialization|  city|revenue|
+---------+-----------+--------------+------+-------+
|     D104| Dr. Suresh|   Orthopedics|Mumbai|  18000|
+---------+-----------+--------------+------+-------+
only showing top 1 row


In [0]:
inner_df.groupBy("specialization")\
.agg(sum("bill_amount").alias("revenue"))\
.orderBy(desc("revenue")).show(1)

+--------------+-------+
|specialization|revenue|
+--------------+-------+
|   Orthopedics|  18000|
+--------------+-------+
only showing top 1 row


In [0]:
inner_df.groupBy("specialization")\
.agg(avg("bill_amount")).show()

+--------------+-----------------+
|specialization| avg(bill_amount)|
+--------------+-----------------+
|   Orthopedics|           9000.0|
|    Cardiology|5833.333333333333|
|    Pediatrics|           1500.0|
|   Dermatology|           1000.0|
|     Neurology|           3500.0|
+--------------+-----------------+



In [0]:
inner_df.groupBy("city")\
.agg(sum("bill_amount")).show()

+---------+----------------+
|     city|sum(bill_amount)|
+---------+----------------+
|    Delhi|            1500|
|  Chennai|            2000|
|Hyderabad|           17500|
|Bangalore|            3500|
|   Mumbai|           18000|
+---------+----------------+



In [0]:
inner_df.groupBy(
    "doctor_name"
).count().show()

+-----------+-----+
|doctor_name|count|
+-----------+-----+
| Dr. Ramesh|    2|
|  Dr. Meera|    1|
| Dr. Suresh|    2|
|  Dr. Anita|    2|
|  Dr. Kiran|    1|
|  Dr. Priya|    1|
+-----------+-----+



In [0]:
revenue_df.orderBy(
    desc("revenue")
).show(3)

+---------+-----------+--------------+---------+-------+
|doctor_id|doctor_name|specialization|     city|revenue|
+---------+-----------+--------------+---------+-------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|
+---------+-----------+--------------+---------+-------+
only showing top 3 rows


In [0]:
performance_df = inner_df.groupBy(
    "doctor_id",
    "doctor_name",
    "specialization"
).agg(
    count("visit_id").alias("visits"),
    sum("bill_amount").alias("revenue")
)

performance_df.show()

+---------+-----------+--------------+------+-------+
|doctor_id|doctor_name|specialization|visits|revenue|
+---------+-----------+--------------+------+-------+
|     D106|  Dr. Kiran|    Cardiology|     1|   7000|
|     D101| Dr. Ramesh|    Cardiology|     2|  10500|
|     D102|  Dr. Priya|     Neurology|     1|   3500|
|     D104| Dr. Suresh|   Orthopedics|     2|  18000|
|     D103|  Dr. Anita|   Dermatology|     2|   2000|
|     D105|  Dr. Meera|    Pediatrics|     1|   1500|
+---------+-----------+--------------+------+-------+



In [0]:
hospital_df = spark.read \
    .option("multiline", "true") \
    .json("file:/Workspace/Users/atsthasneem@gmail.com/hospital_config.json")

hospital_df.printSchema()
hospital_df.show(truncate=False)

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- hospital_id: string (nullable = true)
 |-- hospital_name: string (nullable = true)
 |-- services: array (nullable = true)
 |    |-- element: string (containsNull = true)

+---------+-----------------------------+-----------+----------------+------------------------------------+
|city     |contact                      |hospital_id|hospital_name   |services                            |
+---------+-----------------------------+-----------+----------------+------------------------------------+
|Hyderabad|{apollo@mail.com, 9876500011}|H101       |Apollo Hospital |[Cardiology, Neurology, Dermatology]|
|Hyderabad|{yashoda@mail.com, NULL}     |H102       |Yashoda Hospital|[Cardiology, Orthopedics]           |
|Bangalore|{NULL, 9876500013}           |H103       |Care Hospital   |[Neurology, Pediatrics]             |
+---------+-

In [0]:
from pyspark.sql.functions import col  
hospital_df.select(
    "hospital_name",
      col("contact.phone").alias("phone")

).show()

+----------------+----------+
|   hospital_name|     phone|
+----------------+----------+
| Apollo Hospital|9876500011|
|Yashoda Hospital|      NULL|
|   Care Hospital|9876500013|
+----------------+----------+



In [0]:
hospital_df.select(
    "hospital_name",
    col("contact.email").alias("email")
).show()

+----------------+----------------+
|   hospital_name|           email|
+----------------+----------------+
| Apollo Hospital| apollo@mail.com|
|Yashoda Hospital|yashoda@mail.com|
|   Care Hospital|            NULL|
+----------------+----------------+



In [0]:
hospital_df.filter(
    col("contact.phone").isNull()
).show()

+---------+--------------------+-----------+----------------+--------------------+
|     city|             contact|hospital_id|   hospital_name|            services|
+---------+--------------------+-----------+----------------+--------------------+
|Hyderabad|{yashoda@mail.com...|       H102|Yashoda Hospital|[Cardiology, Orth...|
+---------+--------------------+-----------+----------------+--------------------+



In [0]:
hospital_df.filter(
    col("contact.email").isNull()
).show()

+---------+------------------+-----------+-------------+--------------------+
|     city|           contact|hospital_id|hospital_name|            services|
+---------+------------------+-----------+-------------+--------------------+
|Bangalore|{NULL, 9876500013}|       H103|Care Hospital|[Neurology, Pedia...|
+---------+------------------+-----------+-------------+--------------------+



In [0]:
from pyspark.sql.functions import *
hospital_df = hospital_df.withColumn(
    "phone",
    coalesce(col("contact.phone"),lit("Not Available"))
)

In [0]:
hospital_df.select(
    "hospital_name",
    "phone"
).show()

+----------------+-------------+
|   hospital_name|        phone|
+----------------+-------------+
| Apollo Hospital|   9876500011|
|Yashoda Hospital|Not Available|
|   Care Hospital|   9876500013|
+----------------+-------------+



In [0]:
hospital_df = hospital_df.withColumn(
    "email",
    coalesce(col("contact.email"),lit("Not Available"))
)

In [0]:
hospital_df.select(
    "hospital_name","city"
).show()

+----------------+---------+
|   hospital_name|     city|
+----------------+---------+
| Apollo Hospital|Hyderabad|
|Yashoda Hospital|Hyderabad|
|   Care Hospital|Bangalore|
+----------------+---------+



In [0]:
hospital_df.select(
    "hospital_name",
    col("contact.phone")
).show()

+----------------+----------+
|   hospital_name|     phone|
+----------------+----------+
| Apollo Hospital|9876500011|
|Yashoda Hospital|      NULL|
|   Care Hospital|9876500013|
+----------------+----------+



In [0]:
hospital_df.withColumn(
    "service",
    explode("services")
)

DataFrame[city: string, contact: struct<email:string,phone:string>, hospital_id: string, hospital_name: string, services: array<string>, phone: string, email: string, service: string]

In [0]:
hospital_df.show()

+---------+--------------------+-----------+----------------+--------------------+-------------+----------------+
|     city|             contact|hospital_id|   hospital_name|            services|        phone|           email|
+---------+--------------------+-----------+----------------+--------------------+-------------+----------------+
|Hyderabad|{apollo@mail.com,...|       H101| Apollo Hospital|[Cardiology, Neur...|   9876500011| apollo@mail.com|
|Hyderabad|{yashoda@mail.com...|       H102|Yashoda Hospital|[Cardiology, Orth...|Not Available|yashoda@mail.com|
|Bangalore|  {NULL, 9876500013}|       H103|   Care Hospital|[Neurology, Pedia...|   9876500013|   Not Available|
+---------+--------------------+-----------+----------------+--------------------+-------------+----------------+



In [0]:
hospital_df.withColumn(
    "service",
    explode("services")
).select(
    "hospital_name",
    "service"
).show()

+----------------+-----------+
|   hospital_name|    service|
+----------------+-----------+
| Apollo Hospital| Cardiology|
| Apollo Hospital|  Neurology|
| Apollo Hospital|Dermatology|
|Yashoda Hospital| Cardiology|
|Yashoda Hospital|Orthopedics|
|   Care Hospital|  Neurology|
|   Care Hospital| Pediatrics|
+----------------+-----------+



In [0]:
hospital_df.groupBy("city").count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Hyderabad|    2|
|Bangalore|    1|
+---------+-----+



In [0]:
hospital_df.withColumn(
    "service",
    explode("services")
).groupBy("service").count().show()

+-----------+-----+
|    service|count|
+-----------+-----+
|Orthopedics|    1|
| Cardiology|    2|
| Pediatrics|    1|
|Dermatology|    1|
|  Neurology|    2|
+-----------+-----+



In [0]:
hospital_df.filter(
    array_contains("services","Cardiology")
).show()

hospital_df.filter(
    array_contains("services","Neurology")
).show()

hospital_df.filter(
    array_contains("services","Orthopedics")
).show()

hospital_df.filter(
    array_contains("services","Pediatrics")
).show()

+---------+--------------------+-----------+----------------+--------------------+-------------+----------------+
|     city|             contact|hospital_id|   hospital_name|            services|        phone|           email|
+---------+--------------------+-----------+----------------+--------------------+-------------+----------------+
|Hyderabad|{apollo@mail.com,...|       H101| Apollo Hospital|[Cardiology, Neur...|   9876500011| apollo@mail.com|
|Hyderabad|{yashoda@mail.com...|       H102|Yashoda Hospital|[Cardiology, Orth...|Not Available|yashoda@mail.com|
+---------+--------------------+-----------+----------------+--------------------+-------------+----------------+

+---------+--------------------+-----------+---------------+--------------------+----------+---------------+
|     city|             contact|hospital_id|  hospital_name|            services|     phone|          email|
+---------+--------------------+-----------+---------------+--------------------+----------+-----

In [0]:
from pyspark.sql.functions import *

flattened_df = hospital_df.select(
    "hospital_id",
    "hospital_name",
    "city",
    col("contact.phone").alias("phone"),
    col("contact.email").alias("email"),
    "services"
)

flattened_df.show(truncate=False)

+-----------+----------------+---------+----------+----------------+------------------------------------+
|hospital_id|hospital_name   |city     |phone     |email           |services                            |
+-----------+----------------+---------+----------+----------------+------------------------------------+
|H101       |Apollo Hospital |Hyderabad|9876500011|apollo@mail.com |[Cardiology, Neurology, Dermatology]|
|H102       |Yashoda Hospital|Hyderabad|NULL      |yashoda@mail.com|[Cardiology, Orthopedics]           |
|H103       |Care Hospital   |Bangalore|9876500013|NULL            |[Neurology, Pediatrics]             |
+-----------+----------------+---------+----------+----------------+------------------------------------+



In [0]:
# Q59
# Code is correct but execution may fail in restricted Databricks
# workspaces where DBFS root and Unity Catalog volumes are disabled.

flattened_df.write.mode("overwrite").parquet("hospital_parquet")
#60
flattened_df.write.mode("overwrite")\
.option("header",True)\
.csv("hospital_csv")

---------------------------------------------------------------------------
UnsupportedOperationException             Traceback (most recent call last)
File <command-5343852100210995>, line 5
      1 # Q59
      2 # Code is correct but execution may fail in restricted Databricks
      3 # workspaces where DBFS root and Unity Catalog volumes are disabled.
----> 5 flattened_df.write.mode("overwrite").parquet("hospital_parquet")
      6 #60
      7 flattened_df.write.mode("overwrite")\
      8 .option("header",True)\
      9 .csv("hospital_csv")

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:779, in DataFrameWriter.parquet(self, path, mode, partitionBy, compression)
    777     self.partitionBy(partitionBy)
    778 self._set_opts(compression=compression)
--> 779 self.format("parquet").save(path)

File /databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/readwriter.py:703, in DataFrameWriter.save(self, path, format, mode, partitionBy,

In [0]:
revenue_df = inner_df.groupBy(
    "doctor_id",
    "doctor_name",
    "specialization",
    "city"
).agg(
    sum("bill_amount").alias("revenue")
)

windowSpec = Window.orderBy(desc("revenue"))

In [0]:
revenue_df.withColumn(
    "rank",
    rank().over(windowSpec)
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+----+
|doctor_id|doctor_name|specialization|     city|revenue|rank|
+---------+-----------+--------------+---------+-------+----+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|   1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|   2|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|   3|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|   4|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|   5|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|   6|
+---------+-----------+--------------+---------+-------+----+



In [0]:
revenue_df.withColumn(
    "dense_rank",
    dense_rank().over(windowSpec)
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+----------+
|doctor_id|doctor_name|specialization|     city|revenue|dense_rank|
+---------+-----------+--------------+---------+-------+----------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|         1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|         2|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|         3|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|         4|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|         5|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|         6|
+---------+-----------+--------------+---------+-------+----------+



In [0]:
revenue_df.withColumn(
    "row_num",
    row_number().over(windowSpec)
).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+-------+
|doctor_id|doctor_name|specialization|     city|revenue|row_num|
+---------+-----------+--------------+---------+-------+-------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|      1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|      2|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|      3|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|      4|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|      5|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|      6|
+---------+-----------+--------------+---------+-------+-------+



In [0]:
revenue_df.orderBy(desc("revenue")).show(1)

+---------+-----------+--------------+------+-------+
|doctor_id|doctor_name|specialization|  city|revenue|
+---------+-----------+--------------+------+-------+
|     D104| Dr. Suresh|   Orthopedics|Mumbai|  18000|
+---------+-----------+--------------+------+-------+
only showing top 1 row


In [0]:
revenue_df.orderBy(desc("revenue")).show(3)

+---------+-----------+--------------+---------+-------+
|doctor_id|doctor_name|specialization|     city|revenue|
+---------+-----------+--------------+---------+-------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|
+---------+-----------+--------------+---------+-------+
only showing top 3 rows


In [0]:
specWindow = Window.partitionBy(
    "specialization"
).orderBy(desc("revenue"))

revenue_df.withColumn(
    "rank",
    rank().over(specWindow)
).filter(col("rank")==1).show()

+---------+-----------+--------------+---------+-------+----+
|doctor_id|doctor_name|specialization|     city|revenue|rank|
+---------+-----------+--------------+---------+-------+----+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|   1|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|   1|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|   1|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|   1|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|   1|
+---------+-----------+--------------+---------+-------+----+



In [0]:
revenue_df.withColumn(
    "rank",
    rank().over(specWindow)
).filter(col("rank")<=2).show()

+---------+-----------+--------------+---------+-------+----+
|doctor_id|doctor_name|specialization|     city|revenue|rank|
+---------+-----------+--------------+---------+-------+----+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|   1|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|   2|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|   1|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|   1|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|   1|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|   1|
+---------+-----------+--------------+---------+-------+----+



In [0]:
revenue_df.withColumn(
    "running_revenue",
    sum("revenue").over(windowSpec)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


DataFrame[doctor_id: string, doctor_name: string, specialization: string, city: string, revenue: bigint, running_revenue: bigint]

In [0]:
revenue_df.withColumn(
    "prev_revenue",
    lag("revenue").over(windowSpec)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


DataFrame[doctor_id: string, doctor_name: string, specialization: string, city: string, revenue: bigint, prev_revenue: bigint]

In [0]:
revenue_df.withColumn(
    "next_revenue",
    lead("revenue").over(windowSpec)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


DataFrame[doctor_id: string, doctor_name: string, specialization: string, city: string, revenue: bigint, next_revenue: bigint]

In [0]:
revenue_df.withColumn(
    "difference",
    col("revenue")-
    lag("revenue").over(windowSpec)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


DataFrame[doctor_id: string, doctor_name: string, specialization: string, city: string, revenue: bigint, difference: bigint]

In [0]:
revenue_df.withColumn(
    "difference",
    lead("revenue").over(windowSpec)
    - col("revenue")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


DataFrame[doctor_id: string, doctor_name: string, specialization: string, city: string, revenue: bigint, difference: bigint]

In [0]:
cityWindow = Window.partitionBy(
    "city"
).orderBy(desc("revenue"))

revenue_df.withColumn(
    "rank",
    rank().over(cityWindow)
).filter(col("rank")==1).show()

+---------+-----------+--------------+---------+-------+----+
|doctor_id|doctor_name|specialization|     city|revenue|rank|
+---------+-----------+--------------+---------+-------+----+
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|   1|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|   1|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|   1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|   1|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|   1|
+---------+-----------+--------------+---------+-------+----+



In [0]:
cityWindow2 = Window.partitionBy(
    "city"
).orderBy("revenue")

revenue_df.withColumn(
    "rank",
    rank().over(cityWindow2)
).filter(col("rank")==1).show()

+---------+-----------+--------------+---------+-------+----+
|doctor_id|doctor_name|specialization|     city|revenue|rank|
+---------+-----------+--------------+---------+-------+----+
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|   1|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|   1|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|   1|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|   1|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|   1|
+---------+-----------+--------------+---------+-------+----+



In [0]:
leaderboard = revenue_df.withColumn(
    "rank",
    dense_rank().over(windowSpec)
)

leaderboard.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+----+
|doctor_id|doctor_name|specialization|     city|revenue|rank|
+---------+-----------+--------------+---------+-------+----+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|   1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|   2|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|   3|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|   4|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|   5|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|   6|
+---------+-----------+--------------+---------+-------+----+



In [0]:
doctors_df.createOrReplaceTempView("doctors")

In [0]:
visits_df.createOrReplaceTempView("visits")

In [0]:
hospital_df.createOrReplaceTempView("hospitals")

In [0]:
%sql
SELECT * FROM doctors;

doctor_id,doctor_name,specialization,city,experience_years,consultation_fee
D101,Dr. Ramesh,Cardiology,Hyderabad,12,1500
D102,Dr. Priya,Neurology,Bangalore,10,2000
D103,Dr. Anita,Dermatology,Chennai,8,1000
D104,Dr. Suresh,Orthopedics,Mumbai,15,2500
D105,Dr. Meera,Pediatrics,Delhi,6,1200
D106,Dr. Kiran,Cardiology,Hyderabad,20,3000
D107,Dr. Farhan,Neurology,Pune,5,1800
D108,Dr. Nisha,Dermatology,Kochi,9,1100


In [0]:
%sql
SELECT specialization,COUNT(*)
FROM doctors
GROUP BY specialization;

specialization,COUNT(*)
Orthopedics,1
Cardiology,2
Pediatrics,1
Dermatology,2
Neurology,2


In [0]:
%sql
SELECT city,COUNT(*)
FROM doctors
GROUP BY city;

city,COUNT(*)
Delhi,1
Chennai,1
Kochi,1
Hyderabad,2
Bangalore,1
Pune,1
Mumbai,1


In [0]:
%sql
SELECT d.doctor_name,
SUM(v.bill_amount) revenue
FROM doctors d
JOIN visits v
ON d.doctor_id=v.doctor_id
GROUP BY d.doctor_name;

doctor_name,revenue
Dr. Ramesh,10500
Dr. Meera,1500
Dr. Suresh,18000
Dr. Anita,2000
Dr. Kiran,7000
Dr. Priya,3500


In [0]:
%sql
SELECT specialization,
SUM(bill_amount)
FROM doctors d
JOIN visits v
ON d.doctor_id=v.doctor_id
GROUP BY specialization;

specialization,SUM(bill_amount)
Orthopedics,18000
Cardiology,17500
Pediatrics,1500
Dermatology,2000
Neurology,3500


In [0]:
%sql
SELECT doctor_name,
SUM(bill_amount) revenue
FROM doctors d
JOIN visits v
ON d.doctor_id=v.doctor_id
GROUP BY doctor_name
ORDER BY revenue DESC
LIMIT 5;

doctor_name,revenue
Dr. Suresh,18000
Dr. Ramesh,10500
Dr. Kiran,7000
Dr. Priya,3500
Dr. Anita,2000


In [0]:
%sql
SELECT *
FROM visits
WHERE payment_status='Pending';

visit_id,patient_name,doctor_id,visit_date,diagnosis,bill_amount,payment_status,tax,final_bill
V1003,Amit Kumar,D103,2026-01-11,Skin Allergy,2000,Pending,100.0,2100.0
V1008,Meera Nair,D103,2026-01-13,Skin Allergy,0,Pending,0.0,0.0


In [0]:
%sql
SELECT *
FROM hospitals
WHERE array_contains(
services,'Cardiology'
);

city,contact,hospital_id,hospital_name,services,phone,email
Hyderabad,"List(apollo@mail.com, 9876500011)",H101,Apollo Hospital,"List(Cardiology, Neurology, Dermatology)",9876500011,apollo@mail.com
Hyderabad,"List(yashoda@mail.com, null)",H102,Yashoda Hospital,"List(Cardiology, Orthopedics)",Not Available,yashoda@mail.com


In [0]:
%sql
SELECT *
FROM hospitals
WHERE array_contains(
services,'Neurology'
);

city,contact,hospital_id,hospital_name,services,phone,email
Hyderabad,"List(apollo@mail.com, 9876500011)",H101,Apollo Hospital,"List(Cardiology, Neurology, Dermatology)",9876500011,apollo@mail.com
Bangalore,"List(null, 9876500013)",H103,Care Hospital,"List(Neurology, Pediatrics)",9876500013,Not Available


In [0]:
%sql
SELECT *
FROM hospitals
WHERE contact.phone IS NULL
OR contact.email IS NULL;

city,contact,hospital_id,hospital_name,services,phone,email
Hyderabad,"List(yashoda@mail.com, null)",H102,Yashoda Hospital,"List(Cardiology, Orthopedics)",Not Available,yashoda@mail.com
Bangalore,"List(null, 9876500013)",H103,Care Hospital,"List(Neurology, Pediatrics)",9876500013,Not Available


In [0]:
%sql
SELECT AVG(consultation_fee)
FROM doctors;

AVG(consultation_fee)
1762.5


In [0]:
%sql
SELECT
doctor_name,
COUNT(visit_id) visits,
SUM(bill_amount) revenue
FROM doctors d
JOIN visits v
ON d.doctor_id=v.doctor_id
GROUP BY doctor_name;

doctor_name,visits,revenue
Dr. Ramesh,2,10500
Dr. Meera,1,1500
Dr. Suresh,2,18000
Dr. Anita,2,2000
Dr. Kiran,1,7000
Dr. Priya,1,3500


In [0]:
visits_df = visits_df.fillna({"bill_amount":0})

In [0]:
flattened_df = hospital_df.select(
    "hospital_id",
    "hospital_name",
    "city",
    col("contact.phone").alias("phone"),
    col("contact.email").alias("email")
)

In [0]:
joined_df = doctors_df.join(
    visits_df,
    "doctor_id"
)

In [0]:
revenue_df = joined_df.groupBy(
    "doctor_id"
).agg(
    sum("bill_amount").alias("revenue")
)

In [0]:
ranking_df = revenue_df.withColumn(
    "rank",
    rank().over(windowSpec)
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
spec_summary = joined_df.groupBy(
    "specialization"
).agg(
    sum("bill_amount").alias("revenue"),
    count("visit_id").alias("visits")
)

In [0]:
dashboard_df = joined_df.groupBy(
    "city",
    "specialization"
).agg(
    count("visit_id").alias("total_visits"),
    sum("bill_amount").alias("total_revenue"),
    avg("bill_amount").alias("avg_bill")
)

display(dashboard_df)

city,specialization,total_visits,total_revenue,avg_bill
Hyderabad,Cardiology,3,17500,5833.333333333333
Chennai,Dermatology,2,2000,1000.0
Mumbai,Orthopedics,2,18000,9000.0
Bangalore,Neurology,1,3500,3500.0
Delhi,Pediatrics,1,1500,1500.0
